<a href="https://colab.research.google.com/github/nazri-ai/nazri-ai-lab/blob/main/For_GitHub_Time_Series_Modelling_using_LSTM_23_Jul_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# Title: Python Project: Building a Real-Time Stock Market Price Prediction System
# Author: Abhishek Shaw
# Reproduced by: [Nazri Bamadhaj] (20 Mar 2026)
# Changes: None
# Source: https://medium.com/@abhishekshaw020/python-project-building-a-real-time-stock-market-price-prediction-system-6ce626907342

# In this blog, we’ll walk through building a Real-Time Stock Market Price Prediction System using various data science and
# machine learning libraries like Plotly, NumPy, SciPy, Scikit-learn, and TensorFlow. This project will provide a comprehensive
# understanding of how to use these libraries together in a practical scenario.

# 1. Project Overview
# This project aims to predict future stock prices based on historical data. We’ll use LSTM (Long Short-Term Memory), a type of
# Recurrent Neural Network (RNN) commonly used in time-series prediction.

# We will:
# Fetch real-time stock data
# Perform exploratory data analysis (EDA)
# Visualize data using Plotly
# Build an LSTM model using TensorFlow for prediction
# Evaluate model performance

In [24]:
# 2. Environment Setup
# Para 1 - install the required libraries:

!pip install plotly numpy scipy scikit-learn tensorflow yfinance

In [25]:
# 3. Fetching Real-Time Stock Data
# We’ll use the yfinance package to download stock data for any company. For this project, let’s predict stock prices for Tesla (TSLA).
# The dataset will include columns like Open, High, Low, Close, Adj Close, and Volume.

import yfinance as yf
import pandas as pd

# Fetch historical stock data for Tesla
stock_data = yf.download('TSLA', start='2016-10-01', end='2024-10-01')

# Display the first few rows of the dataset
stock_data.head()

/tmp/ipykernel_1259/1326096359.py:9: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,TSLA,TSLA,TSLA,TSLA,TSLA
Date,,,,,
2016-10-03,14.246667,14.378000,13.883333,14.153333,89998500
2016-10-04,14.094000,14.221333,13.921333,14.206667,53122500
2016-10-05,13.897333,14.210000,13.874667,14.149333,28162500
2016-10-06,13.400000,13.614000,13.347333,13.497333,70551000
2016-10-07,13.107333,13.421333,13.053333,13.400000,52395000


In [26]:
# 4. Data Preprocessing
# We’ll preprocess the data by scaling the features and splitting it into training and testing sets.

import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Use only the 'Close' column for price prediction
close_prices = stock_data['Close'].values

# Normalize the dataset using MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(close_prices.reshape(-1, 1))

# Split the data into training (80%) and testing (20%) sets
train_size = int(len(scaled_data) * 0.8)
train_data, test_data = scaled_data[:train_size], scaled_data[train_size:]


In [27]:
# 5. Data Preparation for LSTM
# To train the LSTM model, we need to create sequences of past stock prices as features and the next stock price as the target.

def create_sequences(data, seq_length):
    x, y = [], []
    for i in range(seq_length, len(data)):
        x.append(data[i-seq_length:i, 0])
        y.append(data[i, 0])
    return np.array(x), np.array(y)

# Create sequences from the training and test data
seq_length = 60  # Use the last 60 days to predict the next day's price
x_train, y_train = create_sequences(train_data, seq_length)
x_test, y_test = create_sequences(test_data, seq_length)

# Reshape the input data to be compatible with LSTM
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

In [28]:
# 6. Building the LSTM Model (Baseline LSTM)
# Now, we’ll build the LSTM model using TensorFlow and Keras.

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Initialize the model
model = Sequential()

# Add LSTM layers
model.add(LSTM(units=100, return_sequences=True, input_shape=(x_train.shape[1], 1)))
model.add(Dropout(0.2))

model.add(LSTM(units=100, return_sequences=False))
model.add(Dropout(0.2))

# Add output layer
model.add(Dense(units=1))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
model.fit(x_train, y_train, epochs=20, batch_size=32)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Epoch 1/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 79ms/step - loss: 0.0097
Epoch 2/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step - loss: 0.0023
Epoch 3/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 4s 81ms/step - loss: 0.0020
Epoch 4/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - loss: 0.0017
Epoch 5/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step - loss: 0.0015
Epoch 6/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 4s 78ms/step - loss: 0.0016
Epoch 7/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 4s 80ms/step - loss: 0.0016
Epoch 8/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step - loss: 0.0014
Epoch 9/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 4s 81ms/step - loss: 0.0017
Epoch 10/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 6s 94ms/step - loss: 0.0012
Epoch 11/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 5s 87ms/step - loss: 0.0014
Epoch 12/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - loss: 0.0014
Epoch 13/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 7s 110ms/step - loss: 0.0013
Epoch 14/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 4s 78ms/step - loss: 0.0012
Epoch 15/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - loss: 0.0012

In [29]:
# 7. Making Predictions

# Once the model is trained, we can use it to predict stock prices on the test data.

# Predict stock prices on the test data
predictions = model.predict(x_test)

# Inverse transform the predictions back to original price scale
predictions = scaler.inverse_transform(predictions)

# Inverse transform the actual test data
y_test_scaled = scaler.inverse_transform(y_test.reshape(-1, 1))

11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step


In [31]:
# 8. Visualizing the Results with Plotly

# We’ll use Plotly to visualize the actual vs. predicted stock prices.
# This interactive plot will display the comparison between actual and predicted stock prices.

import plotly.graph_objs as go

# Create a plotly figure
fig = go.Figure()

# Add trace for actual prices
fig.add_trace(go.Scatter(x=stock_data.index[-len(y_test):], y=y_test_scaled.flatten(), mode='lines', name='Actual Price'))

# Add trace for predicted prices
fig.add_trace(go.Scatter(x=stock_data.index[-len(y_test):], y=predictions.flatten(), mode='lines', name='Predicted Price'))

# Add titles and labels
fig.update_layout(title='Tesla Stock Price Prediction', xaxis_title='Date', yaxis_title='Stock Price (USD)')

# Show the figure
fig.show()

In [32]:
# 9. Model Evaluation
# We’ll evaluate the model’s performance using Mean Squared Error (MSE) and Root Mean Squared Error (RMSE).

from sklearn.metrics import mean_squared_error

# Calculate MSE and RMSE
mse = mean_squared_error(y_test_scaled, predictions)
rmse = np.sqrt(mse)

print(f'Mean Squared Error: {mse}')
print(f'Root Mean Squared Error: {rmse}')

Mean Squared Error: 111.85008578746421
Root Mean Squared Error: 10.5759200917681


In [33]:
# 10. Deploying the Model
# Once the model is built and tested, you can deploy it using a web framework like Flask or Django.
# This allows users to input a stock symbol and get real-time predictions.

# Conclusion
# In this blog, we created a Real-Time Stock Market Price Prediction System using a combination of Plotly, NumPy, SciPy, Scikit-learn,
# and TensorFlow. We fetched real-time stock data, performed data preprocessing, built an LSTM model for time-series prediction,
# and visualized the results using Plotly. This project is not only a great way to learn the fundamentals of time-series forecasting,
# but it also gives you experience with several advanced machine learning libraries. You can further enhance this project by:

# Implementing Grid Search to optimize hyperparameters.
# Using Ensemble Models to combine predictions from multiple models.
# Adding sentiment analysis based on news articles to improve predictions.
# With this knowledge, you’re well on your way to building powerful predictive models in the world of finance!
